In [ ]:
import os, sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from src.spark_session import get_spark_session
from src.config import RAW_DATASET_PATH, CLEANED_DATASET_CSV

spark = get_spark_session(app_name='Zomato_Processing_Notebook')
print('Spark Version:', spark.version)
print('Spark Master:', spark.sparkContext.master)
print('Default Parallelism:', spark.sparkContext.defaultParallelism)

In [ ]:
from src.data_loader import load_raw_data_spark

raw_df = load_raw_data_spark(spark, RAW_DATASET_PATH)
print(f'Total Raw Spark Rows: {raw_df.count():,}')
raw_df.printSchema()
raw_df.select('name', 'location', 'rate', 'votes', 'approx_cost(for two people)').show(5, truncate=False)

In [ ]:
# Create RDD from DataFrame rows
rdd = raw_df.rdd
print('Initial RDD Partitions:', rdd.getNumPartitions())

# 1. RDD filter() transformation: Filter for rows with valid location
valid_loc_rdd = rdd.filter(lambda row: row['location'] is not None and row['location'] != '')

# 2. RDD map() transformation: Create (location, 1) key-value pairs
location_pairs = valid_loc_rdd.map(lambda row: (row['location'].strip(), 1))

# 3. RDD reduceByKey() transformation: Aggregate counts per locality
locality_counts_rdd = location_pairs.reduceByKey(lambda a, b: a + b)

# 4. RDD Action: takeOrdered() to fetch top 5 localities
print('Top 5 Localities via RDD reduceByKey:')
for loc, count in locality_counts_rdd.takeOrdered(5, key=lambda x: -x[1]):
    print(f'  - {loc}: {count:,}')

In [ ]:
# flatMap() decomposes comma-delimited cuisine strings into individual cuisine tokens
cuisine_tokens_rdd = (
    rdd.filter(lambda row: row['cuisines'] is not None)
    .flatMap(lambda row: [c.strip() for c in str(row['cuisines']).split(',') if c.strip()])
    .map(lambda c: (c, 1))
    .reduceByKey(lambda a, b: a + b)
)
print('Top 5 Cuisines via RDD flatMap + reduceByKey:')
for cuisine, count in cuisine_tokens_rdd.takeOrdered(5, key=lambda x: -x[1]):
    print(f'  - {cuisine}: {count:,}')

In [ ]:
from src.data_cleaning import clean_dataset_spark

cleaned_df, quality_audit = clean_dataset_spark(raw_df, deduplicate=True)
print('Cleaning Audit Summary:')
for k, v in quality_audit.items():
    print(f'  {k}: {v}')

cleaned_df.select('name', 'location', 'rating_num', 'approx_cost_num', 'votes_num').show(5)

In [ ]:
# Caching the cleaned DataFrame in memory
cleaned_df.cache()
print('Cleaned DataFrame cached in memory.')

# Inspect physical execution plan via explain()
query = cleaned_df.groupBy('location').count().filter('count > 100')
print('Catalyst Execution Plan (explain):')
query.explain(mode='simple')

In [ ]:
print('Verification of Cleaned Processed Dataset:')
print('Cleaned CSV exists:', CLEANED_DATASET_CSV.exists())
if CLEANED_DATASET_CSV.exists():
    print(f'Size: {CLEANED_DATASET_CSV.stat().st_size / (1024 * 1024):.2f} MB')